<a href="https://colab.research.google.com/github/ThreadsAndKnotsDev/AirLLM-experiment/blob/main/AirLLM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Install dependencies

In [13]:
!pip install -q "transformers>=4.40.0" "accelerate>=0.29.0" airllm torch

# Hard failure due to VRAM Outage

1. Usual imports

In [12]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

2. Define model and baseline resources



In [2]:
MODEL_ID = "Qwen/Qwen2.5-14B-Instruct"

print(f"[+] Total GPU Capacity: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GiB")
print(f"[+] Baseline VRAM: {torch.cuda.memory_allocated() / (1024**2):.2f} MB")

[+] Total GPU Capacity: 14.56 GiB
[+] Baseline VRAM: 0.00 MB


3. Attempt to load the model

In [3]:
try:
    print(f"\n[!] Attempting standard Hugging Face load onto GPU 0...")
    # This tries to allocate the ~28GB parameter graph directly into the 16 GB T4 buffer
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.float16,
        device_map={"": 0}  # Force entire model onto GPU 0
    )
except torch.cuda.OutOfMemoryError as e:
    print("\n" + "="*50)
    print(">>> CRASH CONFIRMED: torch.cuda.OutOfMemoryError")
    print(f">>> Peak VRAM attempted before failure: {torch.cuda.max_memory_allocated() / (1024**3):.2f} GiB")
    print("="*50)
    print("\nRoot Cause:")
    print("Standard PyTorch runtimes treat weight tensors as a single monolithic block.")
    print("A 14B model at 16-bit precision requires ~28 GB VRAM; Colab T4 offers ~14.6 GB.")
finally:
    # Clean up scratchpad memory so Cell 2 starts pristine
    torch.cuda.empty_cache()


[!] Attempting standard Hugging Face load onto GPU 0...


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/579 [00:00<?, ?it/s]


>>> CRASH CONFIRMED: torch.cuda.OutOfMemoryError
>>> Peak VRAM attempted before failure: 14.43 GiB

Root Cause:
Standard PyTorch runtimes treat weight tensors as a single monolithic block.
A 14B model at 16-bit precision requires ~28 GB VRAM; Colab T4 offers ~14.6 GB.


4. Clear occupied VRAM

In [14]:
import torch
import gc

gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

print(f"[+] Reclaimed VRAM: {torch.cuda.memory_allocated() / (1024**2):.2f} MB allocated")

[+] Reclaimed VRAM: 0.00 MB allocated


5. Trying to load model using AirLLM

In [16]:
from airllm import AutoModel

print(f"[+] Baseline VRAM: {torch.cuda.memory_allocated() / (1024**2):.2f} MB")

model = AutoModel.from_pretrained(
    MODEL_ID,
    profiling_mode=True,   # Logs disk I/O vs compute latency per layer
    delete_original=True   # Drops downloaded chunks to keep disk safe
)

print(f"[+] Graph instantiated on PyTorch meta device.")
print(f"[+] VRAM used after graph setup: {torch.cuda.memory_allocated() / (1024**2):.2f} MB")

[+] Baseline VRAM: 0.00 MB
using generic AirLLM streaming model for architecture: Qwen2ForCausalLM


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]



  0%|          | 0/51 [00:00<?, ?it/s]

Loading shard 1/8
saved as: /root/.cache/huggingface/hub/models--Qwen--Qwen2.5-14B-Instruct/snapshots/cf98f3b3bbb457ad9e2bb7baf9a0125b6b88caa8/splitted_model/model.embed_tokens.safetensors


...

saved as: /root/.cache/huggingface/hub/models--Qwen--Qwen2.5-14B-Instruct/snapshots/cf98f3b3bbb457ad9e2bb7baf9a0125b6b88caa8/splitted_model/model.layers.46.safetensors
deleting original file: /root/.cache/huggingface/hub/models--Qwen--Qwen2.5-14B-Instruct/snapshots/cf98f3b3bbb457ad9e2bb7baf9a0125b6b88caa8/model-00007-of-00008.safetensors
Loading shard 8/8
saved as: /root/.cache/huggingface/hub/models--Qwen--Qwen2.5-14B-Instruct/snapshots/cf98f3b3bbb457ad9e2bb7baf9a0125b6b88caa8/splitted_model/model.layers.47.safetensors




 96%|█████████▌| 49/51 [06:58<00:19,  9.98s/it]

 98%|█████████▊| 50/51 [06:59<00:07,  7.11s/it]

saved as: /root/.cache/huggingface/hub/models--Qwen--Qwen2.5-14B-Instruct/snapshots/cf98f3b3bbb457ad9e2bb7baf9a0125b6b88caa8/splitted_model/model.norm.safetensors
saved as: /root/.cache/huggingface/hub/models--Qwen--Qwen2.5-14B-Instruct/snapshots/cf98f3b3bbb457ad9e2bb7baf9a0125b6b88caa8/splitted_model/lm_head.safetensors




100%|██████████| 51/51 [07:24<00:00,  8.71s/it]


built empty Qwen2ForCausalLM via AutoModelForCausalLM (attn=sdpa)
[+] Graph instantiated on PyTorch meta device.
[+] VRAM used after graph setup: 0.00 MB


6. Inference execution and performance

In [21]:
import time

prompt = ["Complete this sentence with one word: In systems architecture, the fastest memory tier is the CPU"]
inputs = model.tokenizer(prompt, return_tensors="pt")

t_start = time.perf_counter()

output = model.generate(
    inputs['input_ids'].cuda(),
    max_new_tokens=1,
    use_cache=False
)

t_total = time.perf_counter() - t_start
tokens_generated = 1
tok_sec = tokens_generated / t_total

print("\n" + "="*50)
print(f">>> Completion: {model.tokenizer.decode(output[0])}")
print(f">>> Peak VRAM Resident: {torch.cuda.max_memory_allocated() / (1024**2):.2f} MB")
print(f">>> Throughput: {tok_sec:.2f} tokens/sec (Total Time: {t_total:.2f}s)")
print("="*50)


>>> Completion: Complete this sentence with one word: In systems architecture, the fastest memory tier is the CPU cache
>>> Peak VRAM Resident: 1501.66 MB
>>> Throughput: 0.01 tokens/sec (Total Time: 124.93s)
